In [1]:
"""
Minion Health Bar Preview Script
=================================
Generates a single PNG showing all minion bar types at multiple HP levels
so you can visually verify they look correct before adding them to DataCreator.

Output: minion_preview.png  (saved next to this script)
"""

import cv2
import numpy as np
import random
from pathlib import Path

# ── Paths (same BASE_DIR as DataCreator) ────────────────────────────────────
BASE_DIR   = Path(r"C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar")
FRAMES_DIR = BASE_DIR / "Classified" / "Frames"
COLORS_DIR = BASE_DIR / "Classified" / "Colors"
OUT_PATH   = BASE_DIR / "minion_preview.png"

# ── Minion frame configs ─────────────────────────────────────────────────────
MINION_FRAMES = {
    "minion_melee":  {"file": "sprite_000.png", "hp_x": 2, "hp_y": 1, "hp_w": 62,  "hp_h": 5},
    "minion_ranged": {"file": "sprite_002.png", "hp_x": 2, "hp_y": 1, "hp_w": 88,  "hp_h": 5},
    "minion_cannon": {"file": "sprite_001.png", "hp_x": 2, "hp_y": 1, "hp_w": 143, "hp_h": 5},
}

# HP percentages to preview
HP_LEVELS = [100, 75, 50, 25, 10]

# Teams to preview
TEAMS = ["ally", "enemy"]

# ── Asset loader ─────────────────────────────────────────────────────────────
def load_color(filename):
    p = COLORS_DIR / filename
    img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"  ⚠ Could not load color asset: {p}")
    return img

ally_hp  = load_color("ally_hpcolor.png")
enemy_hp = load_color("enemy_hp_color.png")

COLOR_ASSETS = {"ally": ally_hp, "enemy": enemy_hp}


# ── Alpha blend helper (verbatim from original DataCreator) ──────────────────
def blend_alpha(dst_roi, src):
    if src is None or src.shape[0] == 0 or src.shape[1] == 0:
        return
    h_dst, w_dst = dst_roi.shape[:2]
    src_r = cv2.resize(src, (min(src.shape[1], w_dst), h_dst),
                       interpolation=cv2.INTER_NEAREST)
    if src_r.shape[2] == 4:
        alpha = src_r[:, :, 3:4] / 255.0
        for c in range(3):
            dst_roi[:, :src_r.shape[1], c] = (
                src_r[:, :, c] * alpha[:, :, 0] +
                dst_roi[:, :src_r.shape[1], c] * (1 - alpha[:, :, 0])
            ).astype(np.uint8)
        dst_roi[:, :src_r.shape[1], 3] = np.maximum(
            dst_roi[:, :src_r.shape[1], 3], src_r[:, :, 3]
        )
    else:
        dst_roi[:, :src_r.shape[1], :3] = src_r[:, :, :3]
        dst_roi[:, :src_r.shape[1], 3]  = 255


# ── Auto-punch alpha: make the near-black interior transparent ───────────────
def punch_alpha(frame_bgr):
    """
    Converts a 3-channel frame PNG into BGRA by detecting the dark interior
    fill region and setting it transparent.

    The frame border pixels are slightly brighter/different from the near-black
    interior, but both are very dark. Strategy:
    - Row 0 and last row are pure border → always opaque
    - Col 0 and last col are border → always opaque
    - Interior pixels that are near-black (all channels < 20) → transparent
    - Pixels that vary between channels (actual border chrome) → opaque
    """
    h, w = frame_bgr.shape[:2]
    bgra = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2BGRA)
    bgra[:, :, 3] = 255  # start fully opaque

    for row in range(1, h - 1):          # skip top/bottom border rows
        for col in range(2, w - 2):      # skip left/right border cols
            px = frame_bgr[row, col].astype(int)
            b, g, r = px
            # Near-black uniform pixels = interior fill slot → punch transparent
            if b < 18 and g < 18 and r < 18:
                bgra[row, col, 3] = 0

    return bgra


# ── Gradient-preserving strip crop (from original overlay_block) ─────────────
def crop_strip_proportional(strip_img, target_w, full_w, target_h):
    """Crop the strip proportionally so gradients don't get squashed."""
    if strip_img is None or target_w <= 0:
        return None

    img = strip_img.copy()

    # Auto-crop dead padding
    if img.shape[2] == 4:
        gray = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2GRAY)
        mask = ((img[:, :, 3] > 20) & (gray > 20)).astype(np.uint8)
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        mask = (gray > 20).astype(np.uint8)
    coords = cv2.findNonZero(mask)
    if coords is not None:
        cx, cy, cw, ch = cv2.boundingRect(coords)
        img = img[:, cx: cx + cw]

    native_w = img.shape[1]
    crop_w   = max(1, min(native_w, int(native_w * (target_w / full_w))))
    cropped  = img[:, :crop_w]
    resized  = cv2.resize(cropped, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
    return resized


# ── Build one minion health bar ───────────────────────────────────────────────
def make_minion_bar(frame_name, team, hp_pct):
    cfg       = MINION_FRAMES[frame_name]
    frame_path = FRAMES_DIR / cfg["file"]

    frame_raw = cv2.imread(str(frame_path), cv2.IMREAD_UNCHANGED)
    if frame_raw is None:
        print(f"  ✗ Could not load frame: {frame_path}")
        return None

    # Convert to BGRA if needed, punch interior transparent
    if frame_raw.shape[2] == 3:
        frame_bgra = punch_alpha(frame_raw)
    else:
        frame_bgra = frame_raw.copy()

    hp_x, hp_y = cfg["hp_x"], cfg["hp_y"]
    hp_w, hp_h = cfg["hp_w"], cfg["hp_h"]

    pad      = 6
    canvas_h = frame_bgra.shape[0] + pad * 2
    canvas_w = frame_bgra.shape[1] + pad * 2
    canvas   = np.zeros((canvas_h, canvas_w, 4), dtype=np.uint8)
    bx, by   = pad, pad

    # 1. Dark HP background slot
    canvas[by + hp_y: by + hp_y + hp_h,
           bx + hp_x: bx + hp_x + hp_w] = (30, 30, 30, 255)

    # 2. HP fill strip
    hp_fill_w  = max(0, int(round(hp_w * hp_pct / 100.0)))
    color_asset = COLOR_ASSETS.get(team)
    if color_asset is not None and hp_fill_w > 0:
        strip = crop_strip_proportional(color_asset, hp_fill_w, hp_w, hp_h)
        if strip is not None:
            # Ensure BGRA
            if strip.shape[2] == 3:
                strip = cv2.cvtColor(strip, cv2.COLOR_BGR2BGRA)
                strip[:, :, 3] = 255
            roi = canvas[by + hp_y: by + hp_y + hp_h,
                         bx + hp_x: bx + hp_x + hp_fill_w]
            blend_alpha(roi, strip)

    # 3. Punch alpha hole in frame over HP slot so fill shows through
    frame_bgra[hp_y: hp_y + hp_h, hp_x: hp_x + hp_w, 3] = 0

    # 4. Frame chrome on top
    f_roi = canvas[by: by + frame_bgra.shape[0], bx: bx + frame_bgra.shape[1]]
    blend_alpha(f_roi, frame_bgra)

    # 5. Crop to content
    non_zero = cv2.findNonZero(canvas[:, :, 3])
    if non_zero is None:
        return None
    x, y, w, h = cv2.boundingRect(non_zero)
    return canvas[y: y + h, x: x + w]


# ── Compose preview sheet ─────────────────────────────────────────────────────
def build_preview():
    print("Building minion health bar preview...")
    print(f"  Frames dir : {FRAMES_DIR}")
    print(f"  Colors dir : {COLORS_DIR}")
    print()

    # Verify frames exist
    for name, cfg in MINION_FRAMES.items():
        p = FRAMES_DIR / cfg["file"]
        exists = "✓" if p.exists() else "✗ MISSING"
        img = cv2.imread(str(p)) if p.exists() else None
        size = f"{img.shape[1]}x{img.shape[0]}" if img is not None else "n/a"
        print(f"  {exists} {name:16} → {cfg['file']:20} ({size})")
    print()

    rows_data = []   # list of (label, bar_bgra)
    frame_names = list(MINION_FRAMES.keys())

    for team in TEAMS:
        for frame_name in frame_names:
            for hp_pct in HP_LEVELS:
                bar = make_minion_bar(frame_name, team, hp_pct)
                label = f"{team} {frame_name.replace('minion_','')} {hp_pct}%"
                rows_data.append((label, bar))

    # ── Layout ──────────────────────────────────────────────────────────────
    LABEL_W   = 220   # pixels reserved for text label
    BAR_SCALE = 4     # zoom factor so tiny bars are actually visible
    ROW_PAD   = 8
    FONT      = cv2.FONT_HERSHEY_SIMPLEX
    FONT_SCALE = 0.45
    FONT_THICK = 1

    # Compute canvas size
    max_bar_w = 0
    max_bar_h = 0
    for _, bar in rows_data:
        if bar is not None:
            max_bar_w = max(max_bar_w, bar.shape[1] * BAR_SCALE)
            max_bar_h = max(max_bar_h, bar.shape[0] * BAR_SCALE)

    row_h      = max_bar_h + ROW_PAD * 2
    total_h    = row_h * len(rows_data) + 40   # +40 for title
    total_w    = LABEL_W + max_bar_w + 40

    sheet = np.full((total_h, total_w, 3), 40, dtype=np.uint8)  # dark grey bg

    # Title
    cv2.putText(sheet, "Minion Health Bar Preview", (10, 24),
                FONT, 0.65, (220, 220, 220), 1, cv2.LINE_AA)

    y_cursor = 40
    for label, bar in rows_data:
        cy = y_cursor + ROW_PAD

        # Label
        cv2.putText(sheet, label, (8, cy + max_bar_h // 2 + 4),
                    FONT, FONT_SCALE, (200, 200, 200), FONT_THICK, cv2.LINE_AA)

        # Bar (upscaled so it's visible)
        if bar is not None:
            bar_h, bar_w = bar.shape[:2]
            scaled_w = bar_w * BAR_SCALE
            scaled_h = bar_h * BAR_SCALE
            bar_up = cv2.resize(bar, (scaled_w, scaled_h),
                                interpolation=cv2.INTER_NEAREST)

            # Composite BGRA bar onto dark sheet
            x0 = LABEL_W
            y0 = cy
            x1 = min(x0 + scaled_w, total_w)
            y1 = min(y0 + scaled_h, total_h)
            sw = x1 - x0
            sh = y1 - y0

            if bar_up.shape[2] == 4:
                alpha = bar_up[:sh, :sw, 3:4] / 255.0
                for c in range(3):
                    sheet[y0:y1, x0:x1, c] = (
                        bar_up[:sh, :sw, c] * alpha[:, :, 0] +
                        sheet[y0:y1, x0:x1, c] * (1 - alpha[:, :, 0])
                    ).astype(np.uint8)
            else:
                sheet[y0:y1, x0:x1] = bar_up[:sh, :sw, :3]
        else:
            cv2.putText(sheet, "[FAILED]", (LABEL_W + 5, cy + 12),
                        FONT, 0.5, (0, 0, 200), 1, cv2.LINE_AA)

        y_cursor += row_h

    cv2.imwrite(str(OUT_PATH), sheet)
    print(f"Preview saved to: {OUT_PATH}")
    print(f"Canvas size: {total_w}x{total_h}")
    print()
    print("What to check:")
    print("  1. Are the bar outlines (frame chrome) correct?")
    print("  2. Is the blue fill correct for ally bars?")
    print("  3. Is the red fill correct for enemy bars?")
    print("  4. Does HP percentage look proportionally right?")
    print("  5. Are there any white/garbage artifacts?")
    print()
    print("hp_x/hp_y/hp_w/hp_h values used:")
    for name, cfg in MINION_FRAMES.items():
        print(f"  {name}: hp_x={cfg['hp_x']} hp_y={cfg['hp_y']} "
              f"hp_w={cfg['hp_w']} hp_h={cfg['hp_h']}")


if __name__ == "__main__":
    build_preview()

Building minion health bar preview...
  Frames dir : C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar\Classified\Frames
  Colors dir : C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar\Classified\Colors

  ✓ minion_melee     → sprite_000.png       (66x7)
  ✓ minion_ranged    → sprite_002.png       (92x7)
  ✓ minion_cannon    → sprite_001.png       (147x7)

Preview saved to: C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar\minion_preview.png
Canvas size: 848x1360

What to check:
  1. Are the bar outlines (frame chrome) correct?
  2. Is the blue fill correct for ally bars?
  3. Is the red fill correct for enemy bars?
  4. Does HP percentage look proportionally right?
  5. Are there any white/garbage artifacts?

hp_x/hp_y/hp_w/hp_h values used:
  minion_melee: hp_x=2 hp_y=1 hp_w=62 hp_h=5
  minion_ranged: hp_x=2 hp_y=1 hp_w=88 hp_h=5
  minion_cannon: hp_x=2 hp_y=1 hp_w=143 hp_h=5


In [4]:
from pathlib import Path
import cv2

BASE_DIR = Path(r"C:\Users\USER\Documents\GitHub\DeeperHealthBarsLeague\LolHealth-Bar")
FRAMES_DIR = BASE_DIR / "Classified" / "Frames"

for name in ["minion.png", "jungle_mini_1.png", "jungle_mini_2.png"]:
    p = FRAMES_DIR / name
    img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if img is not None:
        print(f"{name}: {img.shape[1]}x{img.shape[0]}px, {img.shape[2]}ch")
    else:
        print(f"{name}: NOT FOUND")

minion.png: 66x7px, 4ch
jungle_mini_1.png: 147x7px, 4ch
jungle_mini_2.png: 92x7px, 4ch
